# SAMGPT 当前预训练流程与内存优化（代码一致版）

本文档对应当前 `src/train/execute.py` 的实现，重点解释两件事：

1. 当前预训练到底如何走（stage1/stage2 到 GRAPHCL/EGRAPHCL）。
2. 为什么会爆内存，以及现在如何解决（在线子图训练、splitGRAPHCL、采样提速）。

---

## 1. 总体流程（从 run.sh 到训练）

入口脚本是 `run.sh`，调用 `src/train/execute.py`。

主流程分为 5 个阶段：

1. 参数解析与环境准备。
2. 确保 stage1/stage2 工件存在（必要时自动构建）。
3. 依据 stage2 结果组织预训练样本（优先在线子图模式）。
4. 执行 GRAPHCL/EGRAPHCL 预训练（含 early stopping）。
5. 用最优预训练模型做下游 few-shot 评测。

---

## 2. stage1 / stage2 在当前代码中的作用

### 2.1 stage1 的作用

stage1 负责子图结构编码和聚类，输出：

- `structural_encodings.csv`
- `cluster_centers.npy`
- `cluster_labels.npy`

这些结果描述了每个子图的结构属性与簇归属。

### 2.2 stage2 的作用

stage2 读取 stage1 结果，计算：

- representative pool（代表池）
- learnability score（novelty/coverage/redundancy 组合）
- rank_desc（可学性排序）

训练阶段使用 stage2 的筛选信息决定“哪些子图进入预训练”。

---

## 3. 当前预训练样本组织（重点）

当前代码支持两种路径：

1. 在线子图训练（推荐，默认开启）
2. block 图回退路径（兼容保留）

### 3.1 在线子图训练（默认）

参数：

- `graphcl_online_subgraph=1`

流程：

1. 调用 `build_stage2_online_subgraph_samples(...)`。
2. 从 stage2 取到 pool + rank 的筛选结果。
3. 用 stage1 一致的子图切分规则重建子图。
4. 每个子图单独形成样本：`(sub_feature, sub_adj, sub_struct_labels)`。
5. GRAPHCL/EGRAPHCL 对每个子图样本直接做增强与对比学习。

这条路径不需要先把所有子图拼成一张超大图，更符合“先筛子图，再子图级学习”的目标。

### 3.2 block 图回退路径

如果关闭在线模式，会走旧兼容路径：

- 先把子图拼接成 block-diagonal 大图
- 再做增强与训练

这条路径仍保留用于兼容，但不是推荐基线。

---

## 4. 为什么以前会爆内存

历史上有两类内存问题：

### 4.1 CPU 内存爆炸（已解决）

原因：对超大稀疏图做 `todense()`，会在 CPU 申请极大矩阵。

解决：

- 增强改为按子图块逐块执行，再 block 拼接，避免全图稠密化。

### 4.2 GPU OOM（当前主要矛盾）

原因：

1. 子图总量过大导致总训练节点数非常高。
2. 一次前向张量太大，或一次性驻留过多样本到 GPU。

解决（当前代码）：

1. **在线子图训练**：逐子图（或小批）前向，而非一口气吃大图。
2. **每数据集上限**：`graphcl_online_max_subgraphs_per_dataset`。
3. **每 epoch 采样上限**：`graphcl_online_samples_per_dataset_per_epoch`。
4. **early stopping 与 epoch 上限可配置**：
   - `pretrain_max_epochs`
   - `pretrain_patience`

这些机制本质上是在“保留筛选语义”的同时控制显存峰值与训练时长。

---

## 5. splitGRAPHCL 在当前体系中的位置

`splitGRAPHCL` 是兼容方案：

- 当需要时，把一个大样本拆成多个 shard，逐 shard 训练。
- 可设置每 epoch 打乱 shard 顺序，缓解分片偏差。

它的作用主要是工程稳健性（防 OOM），而在线子图训练更贴近理论语义。

---

## 6. 关键参数与推荐基线

### 6.1 内存稳定相关

- `graphcl_online_subgraph=1`
- `graphcl_online_max_subgraphs_per_dataset=800`（可先 400~800）
- `graphcl_online_samples_per_dataset_per_epoch=0`（0 表示不截每 epoch；可设 100~300 提速）
- `pretrain_max_epochs`（建议先降到 800~2000）
- `pretrain_patience`（建议 50~120）

### 6.2 速度与效果折中

如果训练太慢，优先改：

1. 降低 `graphcl_online_samples_per_dataset_per_epoch`。
2. 降低 `graphcl_online_max_subgraphs_per_dataset`。
3. 降低 `pretrain_max_epochs`。

推荐先拿“可稳定复现且不爆显存”的基线，再逐步回调参数。

---

## 7. 为什么这条新流程更合理

与“先拼大图再训练”相比，在线子图训练的优势：

1. 与 stage2 的筛选对象一致（都在子图级）。
2. 显存占用可控，便于扩展到大图数据集。
3. 可按数据集/每轮动态采样，训练效率更高。

代价：

- 对比范围更局部，和全量一次对比不完全等价。

实践上这是一个合理工程近似，通常优先换取可训练性与稳定性。

---

## 8. 调试与日志解读建议

建议重点关注日志字段：

1. `stage2-online ... kept_subgraphs=...`
2. `training_nodes_total / training_nodes_unique / dup_ratio`
3. `per-epoch sample cap per dataset = ...`
4. `Loss:[...]` 的下降趋势与 early stop 触发轮次

如果再次 OOM：

1. 先把 `graphcl_online_max_subgraphs_per_dataset` 下调到 400。
2. 再把 `graphcl_online_samples_per_dataset_per_epoch` 设为 100~200。
3. 最后再考虑进一步缩小子图规模或 top_ratio。

---

## 9. 一句话总结

当前版本的基线策略是：

**stage2 负责选子图，预训练直接在子图样本上在线进行，借助上限与采样控制显存和速度，在保证筛选语义的前提下得到可训练、可复现的流程。**

## 10. 多数据集联合预训练（一步一步）

这一节专门回答：不同预训练数据集在当前代码里是怎么一起训练的。

### 10.1 先各自准备，再一起训练

流程不是把所有数据集先合成一个大图再训练，而是：

1. 每个数据集先独立完成 stage1 和 stage2（各自有自己的输出目录和文件）。
2. 每个数据集根据自己的 stage2 结果筛出子图样本。
3. 训练时把这些样本放到同一个训练循环里联合优化。

所以是“样本构建独立，优化过程联合”。

### 10.2 每个数据集都有自己的一套 prompt 参数

模型里有两组按数据集分开的参数：

1. feature prompt（每个数据集一套）
2. structure prompt（每个数据集一套）

训练时会给每个样本带一个 dataset index，模型根据这个 index 选择对应数据集的 prompt 参数。

这意味着：

1. 不同数据集不会共享同一套 prompt。
2. 但它们共享同一个编码器主干（GCN 或 GAT 主干）。

### 10.3 EGRAPHCL 的相似度阈值也是按数据集独立计算

对于 EGRAPHCL：

1. 每个数据集用自己的聚类中心计算 cluster similarity。
2. 软正例阈值和真负例阈值应用在该数据集自己的相似度矩阵上。
3. 不会拿 A 数据集的结构相似度去判断 B 数据集样本。

### 10.4 一个 epoch 内到底发生了什么

在在线子图模式下，一个 epoch 的核心过程是：

1. 遍历当前轮要训练的子图样本（这些样本来自多个数据集）。
2. 对每个子图做两份增强，形成对比学习输入。
3. 按样本对应的数据集 index 选择该数据集的 prompt。
4. 计算每个样本的对比损失。
5. 对所有样本损失求平均，反向传播，更新共享主干 + 各数据集 prompt。

### 10.5 为什么要这样设计

这种设计同时满足三件事：

1. 保留数据集个性：每个数据集有自己 prompt 和自己 stage2 筛选结果。
2. 学到跨数据集共性：共享编码器主干在联合训练中获得更稳健表示。
3. 工程上可控：在线子图训练避免超大拼图导致的显存峰值。

### 10.6 一句话理解

当前多数据集预训练可以理解为：

每个数据集先独立选子图和独立建 prompt，再在同一训练循环里联合更新一个共享编码器主干。

## 11. 不同子图共享 Prompt 的具体计算示例（无结果版）

这一节给一个最小计算流程，只展示怎么计算，不填具体数值结果。

### 11.1 设定

假设有两个预训练数据集：

1. 数据集 A
2. 数据集 B

stage2 筛完后，保留子图样本为：

1. A: $A_1, A_2, A_3$
2. B: $B_1, B_2$

参数按下面三类组织：

1. 共享编码器参数：$\theta$
2. A 的 prompt 参数：$\phi_A$
3. B 的 prompt 参数：$\phi_B$

这里要点是：A 的所有子图共享 $\phi_A$，B 的所有子图共享 $\phi_B$。

### 11.2 单个子图样本的前向（以 $A_i$ 为例）

1. 两种增强视图：

$$
v_i^{(1)} = \mathrm{Aug}_1(A_i),\quad v_i^{(2)} = \mathrm{Aug}_2(A_i)
$$

2. 经过数据集 A 的 prompt 和共享编码器：

$$
z_i^{(1)} = f_{\theta}(P_{\phi_A}(v_i^{(1)})),\quad
z_i^{(2)} = f_{\theta}(P_{\phi_A}(v_i^{(2)}))
$$

3. 正对相似度（真正例）：

$$
s_i^{+} = \mathrm{sim}(z_i^{(1)}, z_i^{(2)})
$$

### 11.3 EGRAPHCL 的真负例筛选（按数据集独立）

对每个候选负样本 $j \ne i$，设结构相似度为 $\sigma_{ij}$。

阈值规则：

1. $\sigma_{ij} \ge \tau_{pos}$：软正例，忽略
2. $\sigma_{ij} \le \tau_{neg}$：真负例，保留
3. $\tau_{neg} < \sigma_{ij} < \tau_{pos}$：软负例，忽略

因此真正进入负项的集合是：

$$
\mathcal{N}^{\mathrm{true}}_i = \{ j \ne i \mid \sigma_{ij} \le \tau_{neg} \}
$$

### 11.4 单样本损失（示意）

$$
\mathcal{L}_i
= -\log
\frac{\exp(s_i^{+}/T)}
{\exp(s_i^{+}/T)+\sum_{j\in\mathcal{N}^{\mathrm{true}}_i}\exp(\mathrm{sim}(z_i^{(1)},z_j)/T)}
$$

说明：被忽略的软正例和软负例都不进分母。

### 11.5 多数据集合并更新

一个 step 混合了 A、B 的多个子图样本，整体损失写为：

$$
\mathcal{L}_{step} = \frac{1}{|\mathcal{B}|}\sum_{i\in\mathcal{B}}\mathcal{L}_i
$$

反向传播时：

1. 若样本来自 A，只更新 $\phi_A$
2. 若样本来自 B，只更新 $\phi_B$
3. 所有样本都会更新共享编码器 $\theta$

即：

$$
\nabla_{\theta}\mathcal{L}_{step} = \sum_{i\in\mathcal{B}}\nabla_{\theta}\mathcal{L}_i
$$

$$
\nabla_{\phi_A}\mathcal{L}_{step} = \sum_{i\in(\mathcal{B}\cap A)}\nabla_{\phi_A}\mathcal{L}_i,\quad
\nabla_{\phi_B}\mathcal{L}_{step} = \sum_{i\in(\mathcal{B}\cap B)}\nabla_{\phi_B}\mathcal{L}_i
$$

### 11.6 结论

不同子图并不是一子图一套 prompt。

当前实现是：

1. 同一数据集内所有子图共享同一套 prompt
2. 不同数据集使用不同 prompt
3. 所有数据集子图共同更新同一个共享编码器